# [2] Naive Trial with Contextual Focusing with EXAONE + koELECTRA (MIL)

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import BalancedSWUnivDaconDataset

from transformers import ElectraForSequenceClassification, ElectraModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from torch.utils.data import DataLoader
from torch import nn, optim
import torch

from sklearn.metrics import roc_auc_score, f1_score

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from copy import deepcopy
import sys
import gc
import re

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

In [ ]:
# project name
PROJECT_NAME = "2_naive_focusing"

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
long_context_model_id = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"
narrow_context_model_id = "beomi/KcELECTRA-base-v2022"

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
narrow_tokenizer = AutoTokenizer.from_pretrained(narrow_context_model_id)
narrow_tokenizer("", return_tensors="pt")

In [ ]:
narrow_tokenizer = AutoTokenizer.from_pretrained(long_context_model_id)
narrow_tokenizer("", return_tensors="pt")

In [ ]:
from typing import Optional

class FocusingFormerForNaiveTextDetection(ElectraForSequenceClassification):
    def __init__(
        self,
        narrow_layer_height: int = 4,
        quantization_config: Optional[BitsAndBytesConfig] = None
    ):
        long = AutoModelForCausalLM.from_pretrained(
            long_context_model_id,
            trust_remote_code=True,
            quantization_config=quantization_config
        ).transformer
        long.h = long.h[:-narrow_layer_height]
        narrow = ElectraModel.from_pretrained(
            narrow_context_model_id,
            trust_remote_code=True,
            quantization_config=quantization_config
        )
        narrow.config.num_labels = 1
        super().__init__(narrow.config)
        classifier = self.classifier
        del self.classifier, self.electra

        embedding = narrow.get_input_embeddings()
        self.cls_embedding = embedding(torch.tensor([2]))
        self.sep_embedding = embedding(torch.tensor([3]))
        long.set_input_embeddings(embedding)
        self.former = long
        for param in self.former.parameters():
            param.requires_grad = False  # Freeze the former model parameters

        self.bridge = nn.Linear(long.config.hidden_size, narrow.config.hidden_size)

        self.focusing = narrow.encoder.layer[-narrow_layer_height:]  # Only use the last n layers
        for param in self.focusing.parameters():
            param.requires_grad = False  # Freeze the focusing model parameters
        self.gradient_checkpointing = False
        self.classifier = classifier

        self.post_init()

    def forward(
        self,
        input_ids: list[torch.LongTensor],
        attention_mask: list[torch.Tensor]
    ) -> torch.Tensor:
        input_lists = []
        cls_positions = []
        for inputs, masks in zip(input_ids, attention_mask):
            input_lists.extend(inputs)
            cls_positions.append(len(input_lists))

        with torch.no_grad():
            processed = self.former(input_ids=torch.cat(input_lists, dim=0).unsqueeze(0), attention_mask=torch.tensor([cls_positions])).last_hidden_state

        possibilities, start_point = [], 0
        for idx, points in enumerate(cls_positions):
            hidden_states = torch.cat([
                cls_embedding.unsqueeze(0),  # [1, 1, hidden_dim]
                processed[:, start_point:points, :],  # [1, seq_len, hidden_dim]  
                sep_embedding.unsqueeze(0)   # [1, 1, hidden_dim]
            ], dim=1)
            print(hidden_states)
            masks = torch.cat([torch.tensor([1]), attention_mask[idx], torch.tensor([1])])
            print(masks)
            for layer_module in self.focusing:
                hidden_states = layer_module(hidden_states, masks)[0]
            possibilities.append(self.classifier(hidden_states))

In [ ]:
try:
    model = FocusingFormerForNaiveTextDetection.from_pretrained(f"./models/{PROJECT_NAME}_final")
except Exception:
    model = FocusingFormerForNaiveTextDetection()
model.to(device)

In [ ]:
def tokenize(batch):
    return narrow_tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False, add_special_tokens=False,
        padding="longest" if len(batch) > 1 else False
    )

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

### Utils

In [ ]:
def to_label(scores, threshold=0.5):
    return [1 if score > threshold else 0 for score in scores]

def accuracy(scores, preds, labels, silent=False):
    correct, true_human, false_human = 0, 0, 0
    for score, pred, label in zip(scores, preds, labels):
        if pred == label:
            if label == 0: true_human += 1
            correct += 1
            if not silent: print(f"INFO: Correct prediction - expected {label}, got {pred} [{score.tolist()}]")
        else:
            if label == 0: false_human += 1
            if not silent: print(f"ERROR: Incorrect prediction - expected {label}, got {pred} [{score.tolist()}]")
    return correct, true_human, false_human

def split_sentence(text):
    sentences = re.split(r'(?<=[.?!])\s*', text)
    return [s for s in sentences if s]

def smart_aggregation(logits, temperature=0.5):
    # [num_sentences, 1] -> [num_sentences]
    logits = logits.squeeze(-1)

    if len(logits) == 1:
        return logits[0].unsqueeze(0)

    k = min(3, len(logits))
    top_k_values = torch.topk(logits, k, dim=0).values
    attention_weights = nn.functional.softmax(top_k_values / temperature, dim=0)
    result = torch.sum(attention_weights * top_k_values, dim=0)

    return result.unsqueeze(0)

In [ ]:
BATCH_SIZE = 1, 1, 1
GRADIENT_ACCUMULATION_STEPS = 32  # real batch

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: x)

In [ ]:
EPOCHS = 10
LEARNING_RATE = 2e-5, 1e-6

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE[0])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1, min_lr=LEARNING_RATE[1])

### Training Loop

In [ ]:
with (
    tqdm(range(EPOCHS), desc="[Running Epochs]") as epochs,
    tqdm(range(len(train_dataset)//BATCH_SIZE[0]), desc="[Training]") as train_progress,
    tqdm(range(len(valid_dataset)//BATCH_SIZE[1]), desc="[Validating]") as valid_progress
):
    for epoch in epochs:
        train_progress.reset()
        train_loss, train_preds, train_labels = [], [], []

        # Train
        model.train()
        for step, (texts, labels) in enumerate(train_loader):
            try:
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                    print("input_ids 최대값:", tokenized['input_ids'].max())
                    print("vocab size:", narrow_tokenizer.vocab_size)
                    print("input_ids dtype:", tokenized['input_ids'].dtype)
                    print("input_ids shape:", tokenized['input_ids'].shape)
                    print("attention_mask:", tokenized['attention_mask'])
                    print("attention_mask unique values:", tokenized['attention_mask'].unique())
                    print("attention_mask dtype:", tokenized['attention_mask'].dtype)
                    print("attention_mask shape:", tokenized['attention_mask'].shape)

                    tokenized['attention_mask'].to(torch.int32).to(device)
                    print("input_ids:", tokenized['input_ids'])
                    tokenized['input_ids'].to(device)
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                print("tokenized")
                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                bag = smart_aggregation(logits)
                loss = criterion(bag, labels.float().to(device)) #* (1.4 if labels[0].item() == 1 else 1)
                train_preds.append(torch.sigmoid(bag).item())
                train_labels.append(labels.item())
                train_loss.append(loss.item())
                loss.backward()

                if (len(train_loss)+1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad()

                train_progress.update(1)
                train_progress.set_description(f"[Training] Step: {step+1}, Loss: {sum(train_loss)/len(train_loss):.6f}, ROCAUC: {roc_auc_score(train_labels, train_preds):.6f}, F1: {f1_score(train_labels, to_label(train_preds)): .6f}")
            except Exception as e:
                print(e, file=sys.stderr)

        # Validate
        model.eval()
        valid_preds, valid_labels = [], []
        corrects, errors, true_human, false_human = 0, 0, 0, 0
        torch.cuda.empty_cache(); gc.collect(); valid_progress.reset()
        for texts, labels in valid_loader:
            try:
                with torch.no_grad():
                    input_ids, attention_masks = [], []
                    for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                        input_ids.append(tokenized['input_ids'].to(device))
                        attention_masks.append(tokenized['attention_mask'].to(device))

                    logits = model(input_ids=input_ids, attention_mask=attention_masks)
                    bag = smart_aggregation(logits)
                    scores = torch.sigmoid(bag)
                    preds = to_label(scores)

                    valid_preds.append(scores[0].item())
                    valid_labels.append(labels[0].item())
                    c, th, fh = accuracy(scores, preds, labels.tolist(), silent=True)
                    corrects += c
                    errors += len(labels) - c
                    true_human += th
                    false_human += fh
            except Exception as e:
                print(e, file=sys.stderr)

            valid_progress.update(1)
            valid_progress.set_description(f"[Validating] Correct: {(corrects+1)/(corrects+errors+1):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}], ROCAUC: {roc_auc_score(valid_labels, valid_preds):.6f}, F1: {f1_score(valid_labels, valid_preds):.6f}")

        model.save_pretrained(f"./models/{PROJECT_NAME}_{epoch}")
        scheduler.step(criterion(torch.tensor(train_loss).to(device), torch.tensor(train_labels).float().to(device)))

### Validation Loop

In [ ]:
with tqdm(valid_loader, desc="[Validating]") as progress:
    corrects, errors, true_human, false_human, valid_preds, valid_labels = 0, 0, 0, 0, [], []
    model.eval()
    for texts, labels in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                bag = smart_aggregation(logits)
                scores = torch.sigmoid(bag)
                preds = to_label(scores)

                valid_preds.append(scores[0].item())
                valid_labels.append(labels[0].item())
                c, th, fh = accuracy(scores, preds, labels.tolist())
                corrects += c
                errors += len(labels) - c
                true_human += th
                false_human += fh
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Validating] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}], ROCAUC: {roc_auc_score(valid_labels, valid_preds):.6f}, F1: {f1_score(valid_labels, valid_preds):.6f}")

### Final Output

In [ ]:
per_titles = {}
for i, row in test_dataset.raw.iterrows():
    if row['title'] not in per_titles:
        per_titles[row['title']] = [row['paragraph_text']]
    else:
        per_titles[row['title']].append(row['paragraph_text'])
test_dataset_bundled = [[split_sentence(pp) for pp in p] for t, p in per_titles.items()]
test_dataset_bundled

In [ ]:
test_dataset_combined, test_dataset_split = [], []
for bundle in test_dataset_bundled:
    sentences, points = [], []
    for paragraph in bundle:
        sentences.extend(paragraph)
        points.append(len(sentences))
    test_dataset_combined.append(sentences)
    test_dataset_split.append(points)

In [ ]:
results = []
with tqdm(test_dataset_combined, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for texts, points in zip(progress, test_dataset_split):
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in texts]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                bags, start_point = [], 0
                for point in points:
                    bags.append(smart_aggregation(logits[start_point:point]))
                    start_point = point
                scores = torch.sigmoid(torch.tensor(bags))
                results.extend(scores.tolist())
                for preds in to_label(scores):
                    if preds == 0:
                        humans += 1
                    else:
                        ais += 1
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Testing] Human: {humans/len(test_dataset):.2%}, Ai: {ais/len(test_dataset):.2%}")

len(results) == len(test_dataset)

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x="generated", kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv(f"./data/submission_{PROJECT_NAME[2:]}.csv", index=False, encoding='utf-8-sig')